# Análisis de Ventas Retail · Notebook 01: Limpieza y preparación de datos

## Contexto
Este proyecto analiza casi 7 años de ventas reales (febrero 2020 – noviembre 2026) de una tienda chilena de productos naturales y saludables que vende a través de varios canales: tienda física, tienda online, apps de delivery (Uber Eats, Rappi, Cornershop) y Mercado Libre.

Los datos fueron **anonimizados** antes del análisis: se eliminaron los datos personales de clientes, se codificaron vendedores y documentos, y los montos fueron escalados por un factor confidencial. Las tendencias y proporciones se mantienen; las cifras absolutas no corresponden a las reales.

## Objetivo de este notebook
Preparar una base de datos confiable para el análisis de negocio, entendiendo su estructura, detectando problemas de calidad y documentando cada decisión de limpieza. El análisis exploratorio se desarrolla en el notebook `02_analisis_exploratorio`.

## Datos
- `ventas_anonimizadas.csv`: una fila por producto vendido (195.257 filas).
- `pagos_anonimizados.csv`: una fila por forma de pago usada en cada documento (74.985 filas).
- Ambas tablas se relacionan por la columna `id_documento`.

## Etapas
1. Carga y primera revisión de los datos
2. Calidad de datos: duplicados, documentos especiales, devoluciones y categorías
3. Limpieza: aplicación de las decisiones tomadas en la etapa 2
4. Definición del canal de venta a partir de la forma de pago
5. Guardar los datos limpios y resumen de decisiones

## 1. Carga y primera revisión de los datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)  # montos sin decimales; las proporciones se muestran como % (x100)

df = pd.read_csv('../data/processed/ventas_anonimizadas.csv',
                 dtype={'sku': str},
                 parse_dates=['fecha'])

print(df.shape)
df.head()

(195257, 29)


,tipo_movimiento,tipo_de_documento,hora,vendedor,tipo_de_entrega,tipo_de_producto_servicio,sku,producto_servicio,variante,marca,precio_de_lista,precio_neto_unitario,precio_bruto_unitario,cantidad,venta_total_neta,total_impuestos,venta_total_bruta,nombre_de_dcto,descuento_neto,descuento_bruto,pct_descuento,costo_neto_unitario,costo_total_neto,margen,pct_margen,id_cliente,id_documento,canal,fecha
0,venta,FACTURA ELECTRÓNICA,11:59:48,Vendedor_01,Entrega inmediata,Sin Tipo,158290191419,Glosa,NaN,NaN,"1,479,158","1,242,990","1,479,158",1,"1,242,990","236,168","1,479,158",NaN,0,0,0,"1,242,990","1,242,990",0,0,C_4894225216,1,Tienda,2020-02-28
1,venta,FACTURA ELECTRÓNICA,12:13:42,Vendedor_01,Entrega inmediata,Sin Tipo,158290277721,Glosa,NaN,NaN,"3,287,018","2,762,200","3,287,018",1,"2,762,200","524,818","3,287,018",NaN,0,0,0,0,0,"2,762,200",1,C_cf3e18ec18,2,Tienda,2020-02-28
2,venta,BOLETA ELECTRÓNICA T,11:21:21,Vendedor_02,Entrega inmediata,con tipo,1559164446117,Brota,Stevia,NaN,"8,770","7,370","8,770",1,"7,370","1,400","8,770",NaN,0,0,0,"4,558","4,558","2,812",0,C_034f32c376,3,Tienda,2020-02-29
3,venta,BOLETA ELECTRÓNICA T,11:34:45,Vendedor_02,Entrega inmediata,con tipo,1538839204743,Psyllium,Orgnico 200 grs. Magmagem,NaN,"12,278","10,318","12,278",1,"10,318","1,960","12,278",NaN,0,0,0,"6,383","6,383","3,935",0,C_73c1454306,4,Tienda,2020-02-29
4,venta,BOLETA ELECTRÓNICA T,12:55:49,Vendedor_02,Entrega inmediata,Sin Tipo,1011007060866,Del Alba Crema de Manos,50ml,NaN,"9,668","8,124","9,668",1,"8,125","1,543","9,668",NaN,0,0,0,"5,034","5,034","3,091",0,C_05194f6ef5,5,Tienda,2020-02-29


**Interpretación:** el dataset tiene 195.257 líneas de venta y 29 columnas. Cada fila es un producto dentro de un documento (boleta, factura o nota de crédito). Ya en las primeras filas aparecen dos señales de alerta que se revisan más adelante: productos llamados "Glosa" con montos muy altos, y una categoría de producto con valores como "Sin Tipo" o "con tipo", que no son categorías reales.

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195257 entries, 0 to 195256
Data columns (total 29 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   tipo_movimiento            195257 non-null  object        
 1   tipo_de_documento          195257 non-null  object        
 2   hora                       195257 non-null  object        
 3   vendedor                   195257 non-null  object        
 4   tipo_de_entrega            194568 non-null  object        
 5   tipo_de_producto_servicio  195257 non-null  object        
 6   sku                        195257 non-null  object        
 7   producto_servicio          195255 non-null  object        
 8   variante                   8435 non-null    object        
 9   marca                      148159 non-null  object        
 10  precio_de_lista            195257 non-null  float64       
 11  precio_neto_unitario       195257 non-null  float64 

**Interpretación:** los tipos de datos se cargaron correctamente: `fecha` como fecha, montos como números y `id_documento` como entero. Dos detalles a considerar:
- `hora` quedó como texto; se convertirá cuando se analicen las ventas por hora del día.
- `cantidad` es decimal porque hay productos vendidos a granel (280 líneas con cantidades fraccionarias).

In [3]:
df[df['tipo_de_entrega'].isna()]['tipo_movimiento'].value_counts()

tipo_movimiento
devolucion    689
Name: count, dtype: int64

**Interpretación:** los 689 nulos de `tipo_de_entrega` corresponden exactamente a las 689 devoluciones. No es un dato faltante: una devolución no tiene entrega asociada, por lo que no requiere imputación.

In [4]:
# % de nulos por columna; .map() da formato de texto con un decimal
(df.isna().mean() * 100).sort_values(ascending=False).map('{:.1f}%'.format)

variante                     95.7%
nombre_de_dcto               76.8%
id_cliente                   71.3%
marca                        24.1%
tipo_de_entrega               0.4%
producto_servicio             0.0%
tipo_movimiento               0.0%
canal                         0.0%
id_documento                  0.0%
pct_margen                    0.0%
margen                        0.0%
costo_total_neto              0.0%
costo_neto_unitario           0.0%
pct_descuento                 0.0%
descuento_bruto               0.0%
descuento_neto                0.0%
venta_total_neta              0.0%
venta_total_bruta             0.0%
total_impuestos               0.0%
tipo_de_documento             0.0%
cantidad                      0.0%
precio_bruto_unitario         0.0%
precio_neto_unitario          0.0%
precio_de_lista               0.0%
sku                           0.0%
tipo_de_producto_servicio     0.0%
vendedor                      0.0%
hora                          0.0%
fecha               

**Interpretación:** no todos los nulos son datos faltantes; varios tienen un significado de negocio:
- `variante` (95,7%): la mayoría de los productos no tiene variantes.
- `nombre_de_dcto` (76,8%): un nulo significa que la venta **no tuvo descuento**.
- `id_cliente` (71,3%): la mayoría de las compras no tiene un cliente identificado, algo habitual en retail. Esto limita cualquier análisis de clientes al 29% de las líneas.
- `marca` (24,1%): este sí es un dato faltante real; se tendrá en cuenta si se analiza por marca.

In [5]:
for col in ['tipo_movimiento', 'tipo_de_documento', 'canal', 'tipo_de_producto_servicio']:
    print(f'\n=== {col} ===')
    print(df[col].value_counts())


=== tipo_movimiento ===
tipo_movimiento
venta         194568
devolucion       689
Name: count, dtype: int64

=== tipo_de_documento ===
tipo_de_documento
BOLETA ELECTRÓNICA T                       191550
FACTURA ELECTRÓNICA                          1841
BOLETA NO AFECTA O EXENTA ELECTRÓNICA T       878
NOTA DE CRÉDITO ELECTRÓNICA                   684
FACTURA MANUAL                                219
COMPROBANTE VENTA PUNTOS                       62
NOTA DE DÉBITO ELECTRÓNICA                     10
FACTURA NO AFECTA O EXENTA ELECTRÓNICA          8
NOTA DE CREDITO MANUAL                          5
Name: count, dtype: int64

=== canal ===
canal
Tienda                 179082
Delivery Uber/Rappi      8588
Delivery Cornershop      7025
Cafetería                 418
Mercado Libre             140
Mayorista                   3
Otro                        1
Name: count, dtype: int64

=== tipo_de_producto_servicio ===
tipo_de_producto_servicio
Alimentación Natural       55701
Sin Tipo           

**Interpretación:**
- **Documentos:** el 98% son boletas. Las notas de crédito (684 electrónicas + 5 manuales = 689) coinciden con las devoluciones. Las facturas corresponden a ventas a empresas y a las glosas.
- **Canal (lista de precios):** clasifica el 92% de las líneas como "Tienda" y solo 140 como Mercado Libre. No refleja los canales reales, por lo que en la sección 4 el canal se construye desde la forma de pago.
- **Categoría:** "Sin Tipo", "Con tipo" y "con tipo" suman más de 71.000 líneas (37%) sin categoría útil. Además, "Con tipo" y "con tipo" son el mismo valor con distinta capitalización.

In [6]:
pagos = pd.read_csv('../data/processed/pagos_anonimizados.csv', parse_dates=['fecha'])
print(pagos.shape)
pagos.head()

(74985, 8)


,id_documento,tipo_de_documento,fecha,forma_pago,grupo_pago,monto_forma_pago,total_documento,vendedor
0,1,FACTURA ELECTRÓNICA,2020-02-28,EFECTIVO,Efectivo,"1,479,158","1,479,158",Vendedor_01
1,2,FACTURA ELECTRÓNICA,2020-02-28,EFECTIVO,Efectivo,"3,287,018","3,287,018",Vendedor_01
2,21,BOLETA ELECTRÓNICA T,2020-03-02,EFECTIVO,Efectivo,"15,192","14,642",Vendedor_02
3,22,BOLETA ELECTRÓNICA T,2020-03-02,EFECTIVO,Efectivo,"4,834","4,834",Vendedor_02
4,23,BOLETA ELECTRÓNICA T,2020-03-02,EFECTIVO,Efectivo,"4,143","4,129",Vendedor_02


**Interpretación:** la tabla de pagos tiene 74.985 filas, una por cada forma de pago usada en un documento. Un 1,7% de los documentos se pagó con más de una forma de pago. En efectivo, el monto registrado es lo que el cliente entregó (por ejemplo, 15.192 para una compra de 14.642); el vuelto aparece como una fila negativa aparte.

## 2. Calidad de los datos
En esta sección solo se **diagnostican** los problemas, sin modificar los datos. Las correcciones se aplican en la sección 3.

### 2.1 Duplicados

In [7]:
duplicados = df.duplicated()
print(f'Filas duplicadas exactas: {duplicados.sum():,}')

Filas duplicadas exactas: 0


**Interpretación:** no hay filas duplicadas exactas. No se requiere acción.

### 2.2 Glosas: documentos que no son venta de productos

In [8]:
glosas = df[df['producto_servicio'] == 'Glosa']
print(f'Líneas de glosa: {len(glosas):,}')
print(f'% de la venta total: {glosas["venta_total_bruta"].sum() / df["venta_total_bruta"].sum():.1%}')
print(glosas['tipo_movimiento'].value_counts())
glosas[['fecha', 'tipo_de_documento', 'venta_total_bruta', 'cantidad']].sort_values('venta_total_bruta', ascending=False).head(10)

Líneas de glosa: 264
% de la venta total: 2.6%
tipo_movimiento
venta         217
devolucion     47
Name: count, dtype: int64


,fecha,tipo_de_documento,venta_total_bruta,cantidad
140985,2024-12-09,FACTURA ELECTRÓNICA,"8,044,584",1
50762,2021-10-04,FACTURA ELECTRÓNICA,"6,068,212",1
50764,2021-10-04,FACTURA ELECTRÓNICA,"6,068,212",1
50763,2021-10-04,FACTURA ELECTRÓNICA,"6,068,212",1
50765,2021-10-04,FACTURA ELECTRÓNICA,"6,068,210",1
22042,2020-10-30,FACTURA ELECTRÓNICA,"5,313,110",1
24524,2020-11-27,FACTURA ELECTRÓNICA,"4,696,794",1
22041,2020-10-30,FACTURA ELECTRÓNICA,"4,696,794",1
24526,2020-11-27,FACTURA ELECTRÓNICA,"4,696,794",1
24525,2020-11-27,FACTURA ELECTRÓNICA,"4,696,794",1


**Interpretación:** las glosas corresponden a facturas por **acuerdos publicitarios**: son ingresos reales del negocio, pero no ventas de productos de la tienda. Son solo 264 líneas (0,1% del total: 217 facturas y 47 notas de crédito) pero representan el 2,6% de la venta, con montos de millones cuando una venta típica es de pocos miles. Si se mantuvieran, distorsionarían indicadores como el ticket promedio o la estacionalidad.

### 2.3 Devoluciones

In [9]:
dev = df[df['tipo_movimiento'] == 'devolucion']
print(f'Líneas de devolución: {len(dev):,}')
print(f'Monto total devuelto: {dev["venta_total_bruta"].sum():,.0f}')
print(dev['tipo_de_documento'].value_counts())
dev[['cantidad', 'venta_total_bruta']].describe()

Líneas de devolución: 689
Monto total devuelto: -125,983,999
tipo_de_documento
NOTA DE CRÉDITO ELECTRÓNICA    684
NOTA DE CREDITO MANUAL           5
Name: count, dtype: int64


,cantidad,venta_total_bruta
count,689,689
mean,-2,"-182,851"
std,6,"800,141"
min,-150,"-8,044,584"
25%,-1,"-44,582"
50%,-1,"-16,559"
75%,-1,"-7,182"
max,0,0


In [10]:
dev_glosa = dev[dev['producto_servicio'] == 'Glosa']
print(f'Devoluciones de glosas: {len(dev_glosa)}')
print(f'Monto: {dev_glosa["venta_total_bruta"].sum():,.0f}')
print(f'% del monto devuelto: {dev_glosa["venta_total_bruta"].sum() / dev["venta_total_bruta"].sum():.1%}')

Devoluciones de glosas: 47
Monto: -106,765,124
% del monto devuelto: 84.7%


**Interpretación:**
- Las devoluciones vienen con **cantidad y monto negativos**, por lo que al sumarlas con las ventas se restan solas y se obtiene la venta neta.
- La devolución más grande (-8.044.584) coincide exactamente con la glosa más grande: hay notas de crédito que **anulan facturas publicitarias**.
- Esas 47 notas de crédito de glosas explican el **85% del monto devuelto**. Las devoluciones reales de productos son 642 líneas, equivalentes a cerca del 0,6% de la venta. Sin esta revisión, la tasa de devolución habría parecido unas 6 veces mayor.

### 2.4 Cobertura de categorías por año

In [11]:
sin_cat = df['tipo_de_producto_servicio'].isin(['Sin Tipo', 'Con tipo', 'con tipo'])
anio = df['fecha'].dt.year

venta_con_cat = df.loc[~sin_cat, 'venta_total_bruta'].groupby(anio[~sin_cat]).sum()
venta_total = df['venta_total_bruta'].groupby(anio).sum()

# % de la venta con categoría
(venta_con_cat / venta_total * 100).map('{:.1f}%'.format)

fecha
2020    33.6%
2021    53.0%
2022    59.0%
2023    71.7%
2024    86.5%
2025    95.4%
2026    97.5%
Name: venta_total_bruta, dtype: object

**Interpretación:** el catálogo de la tienda se reestructuró en 2026, y el sistema asigna la categoría **actual** del producto a todas sus ventas históricas. Los productos descontinuados quedan sin categoría, por eso la cobertura cae en los años antiguos (34% en 2020 vs. 98% en 2026).

**Decisión:** el análisis por categoría se limita a **2024–2026**, donde la cobertura supera el 85% de la venta. Antes de 2024, comparar categorías sería engañoso: una categoría parecería crecer solo porque se le asignaron más productos.

## 3. Limpieza
Se crea un DataFrame nuevo, `df_retail`, para el análisis retail. El DataFrame original `df` se mantiene intacto por si se necesita analizar los datos completos.

### 3.1 Excluir glosas (facturas publicitarias y sus notas de crédito)

In [12]:
df_retail = df[df['producto_servicio'] != 'Glosa'].copy()

print(f'Filas originales: {len(df):,}')
print(f'Filas retail:     {len(df_retail):,}')
print(f'Filas excluidas:  {len(df) - len(df_retail):,}')

Filas originales: 195,257
Filas retail:     194,993
Filas excluidas:  264


**Interpretación:** se excluyen las 264 líneas de glosa: 217 facturas y las 47 notas de crédito que las anulan, ya que ambas tienen "Glosa" como producto.

### 3.2 Líneas sin cantidad ni monto

In [13]:
sin_valor = (df_retail['cantidad'] == 0) & (df_retail['venta_total_bruta'] == 0)
print(f'Líneas con cantidad y monto en cero: {sin_valor.sum()}')

Líneas con cantidad y monto en cero: 0


**Interpretación:** después de excluir las glosas no quedan líneas en cero. Las devoluciones en cero vistas en la sección 2.3 eran notas de crédito de glosas. No se requiere acción.

### 3.3 Unificar categorías

In [14]:
mapa_categorias = {
    'Sin Tipo': 'Sin categoría',
    'Con tipo': 'Sin categoría',
    'con tipo': 'Sin categoría',
    'Suplemento': 'Vitaminas y Suplementos',
}

df_retail['categoria'] = df_retail['tipo_de_producto_servicio'].replace(mapa_categorias)
df_retail['categoria'].value_counts()

categoria
Sin categoría              71386
Alimentación Natural       55701
Vitaminas y Suplementos    34545
Bebidas Naturales          14973
Cosmetica Natural           9294
Plantas y Hierbas           6343
Hogar Saludable             2751
Name: count, dtype: int64

**Interpretación:** los tres valores que no son categorías reales se unifican como "Sin categoría", y "Suplemento" (26 líneas) se integra a "Vitaminas y Suplementos". Se crea la columna nueva `categoria` y se conserva la original para poder comparar.

### 3.4 Productos sin nombre

In [15]:
cols_revision = ['fecha', 'tipo_de_documento', 'sku', 'tipo_movimiento', 'cantidad', 'venta_total_bruta', 'categoria']
df_retail[df_retail['producto_servicio'].isna()][cols_revision]

,fecha,tipo_de_documento,sku,tipo_movimiento,cantidad,venta_total_bruta,categoria
8131,2020-04-30,NOTA DE DÉBITO ELECTRÓNICA,15882187515806,venta,1,"2,762,200",Sin categoría
120874,2024-04-02,NOTA DE DÉBITO ELECTRÓNICA,1712068955184838,venta,1,"2,739,183",Sin categoría


**Interpretación:** las 2 líneas sin nombre son **notas de débito** (documentos que ajustan el monto de una factura anterior) por más de 2,7 millones cada una, de una unidad y sin categoría. No corresponden a ventas de productos de la tienda y siguen el patrón de los documentos publicitarios, por lo que se excluyen con el mismo criterio de la sección 3.1.

In [16]:
df_retail = df_retail[df_retail['producto_servicio'].notna()].copy()
print(f'Filas retail: {len(df_retail):,}')

Filas retail: 194,991


### 3.5 Revisión de montos atípicos

In [17]:
cols = ['fecha', 'tipo_de_documento', 'producto_servicio', 'cantidad', 'venta_total_bruta', 'categoria']
df_retail.nlargest(15, 'venta_total_bruta')[cols]

,fecha,tipo_de_documento,producto_servicio,cantidad,venta_total_bruta,categoria
132817,2024-09-05,BOLETA ELECTRÓNICA T,Kuvings - Extractor de jugos prensado en frío ...,2,"1,693,439",Hogar Saludable
176879,2026-02-17,BOLETA ELECTRÓNICA T,Kuvings - Extractor de jugo auto10 negro,1,"1,049,622",Hogar Saludable
171714,2025-12-01,FACTURA ELECTRÓNICA,Kuvings - Extractor de jugo auto10 negro,1,"944,659",Hogar Saludable
146438,2025-02-21,BOLETA ELECTRÓNICA T,Kuvings - Extractor de jugos prensado en frío ...,1,"923,956",Hogar Saludable
105214,2023-09-07,BOLETA ELECTRÓNICA T,Kuvings - Extractor de jugos prensado en frío ...,1,"911,526",Hogar Saludable
106515,2023-09-25,BOLETA ELECTRÓNICA T,Kuvings - Extractor de jugos prensado en frío ...,1,"911,526",Hogar Saludable
120397,2024-03-26,FACTURA ELECTRÓNICA,Kuvings - Extractor de jugos prensado en frío ...,1,"911,526",Hogar Saludable
135630,2024-10-05,BOLETA ELECTRÓNICA T,Kuvings - Extractor de jugos prensado en frío ...,1,"911,526",Hogar Saludable
160724,2025-07-22,BOLETA ELECTRÓNICA T,Kuvings - Extractor de jugos prensado en frío ...,1,"877,758",Hogar Saludable
164067,2025-08-26,BOLETA ELECTRÓNICA T,Kuvings - Extractor de jugos prensado en frío ...,1,"877,758",Hogar Saludable


**Interpretación:** como el filtro por nombre no garantiza haber capturado todos los registros ajenos a la venta retail, se revisan las líneas de mayor monto. Las 15 corresponden a extractores de jugo Kuvings, productos reales de alto valor, vendidos con boleta o factura. No se detectan otros registros anómalos y se mantienen en el análisis. Kuvings, al ser un producto de ticket alto, tendrá un peso importante en la venta total.

## 4. Definición del canal de venta

### Regla de canales de venta

El canal de la lista de precios no es confiable (clasifica el 92% de las líneas como "Tienda"), por lo que el canal se construye a partir de la forma de pago. Regla definida con conocimiento del negocio:

| Canal | Formas de pago |
|---|---|
| Tienda física | Tarjeta débito y crédito presencial, efectivo |
| Tienda online | Webpay, MercadoPago, Paypal |
| Apps de delivery | Uber Eats, Rappi, Cornershop, Goodmeal, PedidosYa |
| Mercado Libre | Mercado Libre, ML Full |
| Transferencia (mixto) | Transferencia bancaria |
| Otros | Crédito trabajadores, cheque |

- Las transferencias se mantienen como canal separado porque se usan tanto en tienda física como online.
- Las formas de pago de devoluciones y ajustes (vueltos, notas de crédito) no definen canal.
- Si un documento tiene más de una forma de pago, se asigna el canal de la que tiene el monto mayor.

In [18]:
mapa_canal = {
    'TARJETA DEBITO': 'Tienda física', 'TARJETA CREDITO': 'Tienda física', 'EFECTIVO': 'Tienda física',
    'WEBPAY DEBITO': 'Tienda online', 'WEBPAY CREDITO': 'Tienda online',
    'MERCADOPAGO': 'Tienda online', 'PAYPAL': 'Tienda online',
    'UBER EATS': 'Apps de delivery', 'RAPPI': 'Apps de delivery', 'CORNERSHOP': 'Apps de delivery',
    'GOODMEAL': 'Apps de delivery', 'CUPÓN PEDIDOS YA': 'Apps de delivery',
    'MERCADOLIBRE': 'Mercado Libre', 'ML FULL - NO USAR': 'Mercado Libre',
    'TRANSFERENCIA BANCARIA': 'Transferencia (mixto)',
    'CREDITO TRABAJADORES': 'Otros', 'CHEQUE': 'Otros',
}

pagos['canal_venta'] = pagos['forma_pago'].map(mapa_canal)

pagos_validos = pagos[pagos['canal_venta'].notna()].copy()
pagos_validos['monto_abs'] = pagos_validos['monto_forma_pago'].abs()

# regla del monto mayor: para cada documento se conserva el pago de mayor monto
canal_doc = (pagos_validos
             .sort_values('monto_abs', ascending=False)
             .drop_duplicates('id_documento')
             [['id_documento', 'canal_venta']])

df_retail = df_retail.merge(canal_doc, on='id_documento', how='left')
df_retail['canal_venta'] = df_retail['canal_venta'].fillna('Sin canal')

df_retail['canal_venta'].value_counts()

canal_venta
Tienda física            119291
Tienda online             44475
Apps de delivery          15781
Transferencia (mixto)     11516
Mercado Libre              2957
Sin canal                   823
Otros                       148
Name: count, dtype: int64

In [19]:
pd.crosstab(df_retail['canal_venta'], df_retail['tipo_movimiento'])

tipo_movimiento,devolucion,venta
canal_venta,,
Apps de delivery,0,15781
Mercado Libre,0,2957
Otros,0,148
Sin canal,642,181
Tienda física,0,119291
Tienda online,0,44475
Transferencia (mixto),0,11516


**Interpretación:** las 642 devoluciones quedan sin canal, lo esperable porque se pagan con nota de crédito. Solo 181 líneas de venta no reciben canal: la regla cubre el **99,9% de las ventas**.

Comparado con el canal de la lista de precios, el panorama cambia por completo: la tienda online tiene más de 44.000 líneas (cerca del 23%), las apps de delivery casi 16.000 y Mercado Libre casi 3.000, frente a las 140 que mostraba la columna original.

In [20]:
df_retail.loc[df_retail['tipo_movimiento'] == 'devolucion', 'canal_venta'] = 'Devolución'

sin_canal = df_retail[df_retail['canal_venta'] == 'Sin canal']
print(sin_canal['tipo_de_documento'].value_counts())
print(sin_canal['fecha'].dt.year.value_counts().sort_index())

tipo_de_documento
BOLETA ELECTRÓNICA T                       83
COMPROBANTE VENTA PUNTOS                   58
FACTURA ELECTRÓNICA                        28
NOTA DE DÉBITO ELECTRÓNICA                  8
BOLETA NO AFECTA O EXENTA ELECTRÓNICA T     4
Name: count, dtype: int64
fecha
2020    46
2021    33
2022    36
2023     9
2024    25
2025    18
2026    14
Name: count, dtype: int64


**Interpretación:** las devoluciones se etiquetan como "Devolución" para no mezclarlas con ventas sin canal. De las 181 ventas sin canal, 58 son canjes de puntos: cuando un cliente paga con puntos, el sistema no registra forma de pago. El resto (123 líneas) son documentos sin pago registrado, repartidos en todos los años y sin un patrón claro.

In [21]:
puntos = (df_retail['canal_venta'] == 'Sin canal') & (df_retail['tipo_de_documento'] == 'COMPROBANTE VENTA PUNTOS')
df_retail.loc[puntos, 'canal_venta'] = 'Canje de puntos'
df_retail['canal_venta'].value_counts()

canal_venta
Tienda física            119291
Tienda online             44475
Apps de delivery          15781
Transferencia (mixto)     11516
Mercado Libre              2957
Devolución                  642
Otros                       148
Sin canal                   123
Canje de puntos              58
Name: count, dtype: int64

**Interpretación:** los canjes de puntos reciben su propia etiqueta, porque son parte de un programa de fidelización real. Las 123 líneas restantes se mantienen como "Sin canal" por su impacto mínimo (menos del 0,1%).

## 5. Guardar datos limpios

In [22]:
df_retail = df_retail.drop(columns=['canal'])  # se reemplaza por canal_venta, construido desde la forma de pago

df_retail.to_parquet('../data/processed/ventas_limpias.parquet', index=False)
print(f'Guardado: {len(df_retail):,} filas, {df_retail.shape[1]} columnas')

Guardado: 194,991 filas, 30 columnas


### Resumen de decisiones

| Paso | Decisión | Filas afectadas |
|---|---|---|
| Glosas | Excluidas: acuerdos publicitarios, no ventas de productos | 264 |
| Productos sin nombre | Excluidos: mismo patrón que las glosas | 2 |
| Categorías | "Sin Tipo", "Con tipo" y "con tipo" → "Sin categoría"; "Suplemento" → "Vitaminas y Suplementos" | 71.412 |
| Análisis por categoría | Limitado a 2024–2026 por cobertura | — |
| Canal de venta | Construido desde la forma de pago (cubre el 99,9% de las ventas) | 194.991 |
| Columna `canal` original | Eliminada por no reflejar los canales reales | — |

**Resultado:** `ventas_limpias.parquet`, con 194.991 líneas listas para el análisis exploratorio en el notebook 02.